# Simple Autoencoders With Keras

## Importing TensorFlow and the Other Required Libraries

In [ ]:
import keras
import matplotlib.pyplot as plt
import numpy
import scipy.interpolate
import tensorflow as tf

from keras import layers

## Loading MNIST

We'll use slightly different preprocessing than usual: since we want to be able to predict the input values at the output (the principle of autoencoding), we'll simply project these values into $[0, 1]$ instead of $[0, 255]$. Note that we usually don't do that: we normalize by centering on zero and dividing by the standard deviation.

In [ ]:
(X_train, y_train), (X_test, y_test) = keras.datasets.mnist.load_data()
nb_classes = 10
input_dim = 28 * 28
X_train = X_train.reshape(-1, input_dim) / 255.0
X_test = X_test.reshape(-1, input_dim) / 255.0

In [ ]:
X_train.shape

## Creating the Autoencoder

You should be able to create the basic autoencoder model by yourself.

Watch out for the choice of activation and loss functions!

In [ ]:
# Your code here
encoding_dim = 20
encoder = keras.Sequential()
decoder = keras.Sequential()
autoencoder = keras.Sequential()

### Solution

In [ ]:
encoding_dim = 20
encoder = keras.Sequential(name="encoder")
encoder.add(layers.Input(shape=(input_dim,)))
encoder.add(layers.Dense(encoding_dim, activation="relu"))

decoder = keras.Sequential(name="decoder")
decoder.add(layers.Input(shape=(encoding_dim,)))
decoder.add(layers.Dense(input_dim, activation="sigmoid"))

autoencoder = keras.Sequential(name="autoencoder")
autoencoder.add(encoder)
autoencoder.add(decoder)
autoencoder.summary()

autoencoder.compile(optimizer="adam", loss="mean_squared_error")

In [ ]:
encoding_dim = 20
dense_params = dict(activation="relu", kernel_initializer="orthogonal")

encoder = keras.Sequential(name="encoder")
encoder.add(layers.Input(shape=(input_dim,)))
encoder.add(layers.Dense(150, **dense_params))
encoder.add(layers.Dense(150, **dense_params))
encoder.add(layers.Dense(150, **dense_params))
encoder.add(layers.Dense(150, **dense_params))
encoder.add(layers.Dense(50, **dense_params))
encoder.add(layers.Dense(encoding_dim, kernel_initializer="orthogonal", activation="linear"))

decoder = keras.Sequential(name="decoder")
decoder.add(layers.Input(shape=(encoding_dim,)))
decoder.add(layers.Dense(50, **dense_params))
decoder.add(layers.Dense(150, **dense_params))
decoder.add(layers.Dense(150, **dense_params))
decoder.add(layers.Dense(150, **dense_params))
decoder.add(layers.Dense(150, **dense_params))
decoder.add(layers.Dense(input_dim, activation="sigmoid", kernel_initializer="orthogonal"))

autoencoder = keras.Sequential(name="autoencoder")
autoencoder.add(encoder)
autoencoder.add(decoder)

autoencoder.summary(expand_nested=True)

autoencoder.compile(optimizer="adam", loss="mean_squared_error")

## Training

*Write the line that trains your autoencoder:*

- *50 epochs should be enough*
- *Use a batch size of 256*

In [ ]:
# Your code here

### Solution

In [ ]:
autoencoder.fit(X_train, X_train,
                epochs=50,
                batch_size=256,
                validation_split=0.2)

## Prediction on White Noise

For a first use of the decoder, we can look at what it predicts on white noise input.

*Sometimes, the generated image is almost constant despite the random noise given as input. Why is that?*

In [ ]:
white_noise = numpy.random.random_sample((1, encoding_dim))
plt.imshow(decoder.predict(white_noise).reshape(28, 28), cmap="gray_r")
plt.show()

## Encoding the Test Data

With the encoder we got, we can also encode our test data into the learned space of dimension `encoding_dim`.

In [ ]:
codes = encoder.predict(X_test)

In [ ]:
codes.shape

## Computing Each Digit's Centroid in the Coding Space

Another interesting point is to look at where, on average, the examples of a given label land in the coding space.

In [ ]:
means = numpy.vstack([codes[y_test == i].mean(axis=0)
                      for i in range(nb_classes)])
stds = numpy.vstack([codes[y_test == i].std(axis=0)
                     for i in range(nb_classes)])

# Quick look at the test encodings
for i in range(10):
  dimension_stats = [f"{mean:5.2f}±{std:.2f}"
                     for mean, std in zip(means[i], stds[i])]
  print(f"Class {i} {', '.join(dimension_stats)}")

From there, we can check that each centroid is decoded into something plausible by the decoder.

In [ ]:
f, ax = plt.subplots(1, nb_classes, figsize=(1.4 * nb_classes, 2))

centroid_images = decoder.predict(means).reshape(-1, 28, 28)

for i, centroid_image in enumerate(centroid_images):
  ax[i].imshow(centroid_image, cmap="gray_r")
  ax[i].axis("off")
plt.show()

## Walking the Latent Space Between Two Centroids

Now that we know where the centroids of a given digit are in the latent space, we can walk the space between two centroids to better understand how the latent space is structured.

In [ ]:
def latent_walk(start: int, end: int, n: int = 15):
  interpolator = scipy.interpolate.interp1d([0, n - 1],
                                            means[[start, end], :],
                                            axis=0)
  interpolated_codes = interpolator(range(n))
  interpolated_images = decoder.predict(interpolated_codes).reshape(-1, 28, 28)

  f, ax = plt.subplots(1, n, figsize=(n * 1.4, 2))
  for i, interpolated_image in enumerate(interpolated_images):
    ax[i].imshow(interpolated_image, cmap="gray_r")
    ax[i].axis("off")
  plt.show()


latent_walk(7, 6)

## Autoencoding the Whole Test Set

Autoencode the train and test images and store the resulting images in the variables `X_train_pred` and `X_test_pred`

In [ ]:
# Your code here
X_train_pred = X_train
X_test_pred = X_test

### Solution

In [ ]:
X_train_pred = autoencoder.predict(X_train)
X_test_pred = autoencoder.predict(X_test)

## Visual Evaluation of the Performance

In [ ]:
n = 15  # Number of digits we'll display

random_indexes = numpy.random.choice(X_test_pred.shape[0],
                                     size=n,
                                     replace=False)


def display_samples(indices: numpy.ndarray,
                    X: numpy.ndarray,
                    X_pred: numpy.ndarray,
                    y: numpy.ndarray
                   ) -> None:
  _, ax = plt.subplots(2, n, figsize=(n * 1.4, 4))
  for i, index in enumerate(indices):
      # The original on top
      ax[0, i].set_title(y[index])
      ax[0, i].imshow(X[index].reshape(28, 28), cmap="gray_r")
      ax[0, i].axis("off")

      # The reconstruction at the bottom
      ax[1, i].imshow(X_pred[index].reshape(28, 28), cmap="gray_r")
      ax[1, i].axis("off")
  plt.show()


display_samples(random_indexes, X_test, X_test_pred, y_test)

## Anomaly Detection

You can detect anomalies in the data using the reconstruction error: a large error suggests that the example is outside the normal distribution of the data.

In [ ]:
mse = keras.losses.MeanSquaredError(reduction="none")
test_anomalies = (-mse(X_test_pred, X_test).numpy()).argsort()
train_anomalies = (-mse(X_train_pred, X_train).numpy()).argsort()

n = 20

print("Worst reconstructions on train")
display_samples(train_anomalies[:n], X_train, X_train_pred, y_train)

print("Worst reconstructions on test")
display_samples(test_anomalies[:n], X_test, X_test_pred, y_test)

print("Best reconstructions on train")
display_samples(train_anomalies[-n:], X_train, X_train_pred, y_train)

print("Best reconstructions on test")
display_samples(test_anomalies[-n:], X_test, X_test_pred, y_test)

## Your Turn!

Find an ideal value for the encoding size, along with a suitable model, so that your autoencoder has an acceptable compression loss.

What's your compression ratio?